# Vue2 + Vue3 框架 · 第 10 天

## 智慧商城项目：购物车、结算、订单、个人中心与打包

> 本篇整理项目后半段：购物车 Vuex 模块、全选反选、结算台、提交订单、订单管理、个人中心和生产打包优化。

## 一、购物车 Vuex 模块

购物车数据需要在商品详情、购物车、结算和个人中心之间共享，适合放到 Vuex 模块：

```js
// store/modules/cart.js
export default {
  namespaced: true,
  state: () => ({ cartList: [] }),
  mutations: {
    setCartList (state, list) { state.cartList = list },
    updateCount (state, { id, count }) {
      const item = state.cartList.find(item => item.goods_id === id)
      if (item) item.goods_num = count
    },
    remove (state, id) {
      state.cartList = state.cartList.filter(item => item.goods_id !== id)
    }
  },
  getters: {
    totalCount: state => state.cartList.reduce((sum, item) => sum + item.goods_num, 0),
    checkedList: state => state.cartList.filter(item => item.is_check),
    checkedTotal: (state, getters) => getters.checkedList.reduce(
      (sum, item) => sum + Number(item.goods_price) * item.goods_num, 0
    )
  }
}
```

在根 store 中注册：

```js
modules: { user, cart }
```

## 二、购物车页面功能

### 2.1 渲染和计算

```js
computed: {
  ...mapState('cart', ['cartList']),
  ...mapGetters('cart', ['totalCount', 'checkedList', 'checkedTotal']),
  isAllChecked () {
    return this.cartList.length > 0 && this.cartList.every(item => item.is_check)
  }
}
```

每行使用 `:key="item.goods_id"`，图片、名称、价格和数量来自接口数据。空数组时显示空购物车图片和“去逛逛”按钮。

### 2.2 全选反选

```js
methods: {
  toggleAll (checked) {
    this.cartList.forEach(item => {
      item.is_check = checked
      this.$store.dispatch('cart/updateChecked', { id: item.goods_id, checked })
    })
  }
}
```

更好的做法是封装一个批量接口或 mutation，保证全选操作是一个明确的业务动作。

### 2.3 修改数量和删除

数量框提交新值后调用接口并更新 store；删除成功后重新拉取列表或从 store 移除对应项。所有金额都使用数字计算，显示时再格式化为两位小数。

## 三、购物车编辑状态

编辑态可以控制批量删除和选择框显示：

```js
data () {
  return { isEdit: false }
},
methods: {
  async toggleEdit () {
    this.isEdit = !this.isEdit
  },
  async removeChecked () {
    const ids = this.checkedList.map(item => item.goods_id)
    if (!ids.length) return this.$toast('请选择商品')
    await deleteCart(ids)
    await this.getCartList()
    this.isEdit = false
  }
}
```

不要把编辑状态写到每个商品对象里，页面状态和接口数据分开更容易维护。

## 四、订单结算台

结算页有两种模式：购物车结算和立即购买结算。通过路由 query 区分：

```js
this.$router.push({
  path: '/pay',
  query: { mode: 'buyNow', goodsId, goodsSkuId, goodsNum }
})
```

页面计算属性读取参数：

```js
computed: {
  mode () { return this.$route.query.mode || 'cart' },
  cartIds () { return this.$route.query.cartIds || '' },
  goodsId () { return this.$route.query.goodsId },
  goodsSkuId () { return this.$route.query.goodsSkuId },
  goodsNum () { return this.$route.query.goodsNum }
}
```

根据模式调用订单预览接口：

```js
async getOrderList () {
  const params = this.mode === 'cart'
    ? { cartIds: this.cartIds }
    : { goodsId: this.goodsId, goodsSkuId: this.goodsSkuId, goodsNum: this.goodsNum }
  const { data } = await checkOrder(this.mode, params)
  this.order = data.order
  this.personal = data.personal
}
```

## 五、地址、买家留言和提交订单

地址选择组件通常在结算页通过 `v-model` 或弹层完成，留言直接绑定表单：

```vue
<textarea v-model.trim="remark" placeholder="选填：买家留言（50字内）"></textarea>
```

统一提交订单接口：

```js
export const submitOrder = (mode, params) => request.post('/checkout/submit', {
  mode,
  delivery: 10,
  couponId: 0,
  payType: 10,
  isUsePoints: 0,
  ...params
})
```

提交后：

```js
async submit () {
  const params = this.mode === 'cart'
    ? { remark: this.remark, cartIds: this.cartIds }
    : { remark: this.remark, goodsId: this.goodsId, goodsSkuId: this.goodsSkuId, goodsNum: this.goodsNum }
  await submitOrder(this.mode, params)
  this.$toast.success('支付成功')
  this.$router.replace('/myorder')
}
```

提交按钮要防重复点击，失败时保留用户输入并提示原因。

## 六、订单管理

订单页用 Tab 切换状态：

```vue
<van-tabs v-model="active" sticky>
  <van-tab name="all" title="全部" />
  <van-tab name="payment" title="待支付" />
  <van-tab name="delivery" title="待发货" />
  <van-tab name="received" title="待收货" />
  <van-tab name="comment" title="待评价" />
</van-tabs>
```

监听当前 tab 请求对应数据：

```js
watch: {
  active: {
    immediate: true,
    handler () { this.getOrderList() }
  }
}
```

`OrderListItem` 接收一个订单对象，通过 `v-for` 渲染商品，按照订单状态显示“付款、取消、确认收货、评价”等操作。订单商品总数量可以在获取数据后由订单中的商品数量累加得到。

## 七、个人中心与退出登录

个人中心根据 token 区分已登录和未登录状态：

```js
computed: {
  isLogin () { return Boolean(this.$store.getters['user/token']) }
},
created () {
  if (this.isLogin) this.getUserInfoDetail()
}
```

订单入口可以携带筛选参数：

```js
this.$router.push('/myorder?dataType=payment')
```

退出时清空用户信息和购物车：

```js
async logout () {
  await this.$dialog.confirm({ title: '温馨提示', message: '确认退出吗？' })
  await this.$store.dispatch('user/logout')
  this.$router.replace('/login')
}
```

登出 action 中清除 token、用户信息、本地存储和购物车模块，避免下一个账号看到上一个账号的数据。

## 八、项目打包与上线优化

脚手架只服务于开发阶段，上线使用打包后的 `dist`：

```bash
npm run build
```

### 8.1 `publicPath`

部署到非根路径时，按项目服务器配置设置：

```js
// vue.config.js
module.exports = { publicPath: './' }
```

### 8.2 路由懒加载

把页面拆成异步组件，访问到对应路由时才加载：

```js
const ProDetail = () => import('@/views/prodetail')
const Pay = () => import('@/views/pay')
const MyOrder = () => import('@/views/myorder')
```

懒加载能减小首屏包体积；公共组件和首屏核心页面按需加载，避免过度拆分造成请求过多。

## 九、项目验收清单

- 购物车数量、选中状态和总价在多个页面保持一致。
- 结算页能区分购物车结算和立即购买。
- 提交订单后跳转订单列表，订单状态和商品数量正确。
- 未登录操作会弹确认框，登录后能回到原页面。
- 退出登录后 token、用户信息和购物车都清空。
- `npm run build` 成功，路由懒加载资源能正常获取。